# IY029: Supervised TransformerClassifier — Pairwise Same/Different Task

Trains a `TransformerClassifier` **from scratch** on the IY011/IY014 pairwise same/different task as a supervised baseline, for direct comparison with the SimCLR + SVM results in `IY029_simclr_svm_pairwise.ipynb`.

## Architecture
- **Single model**: `TransformerClassifier(input_size=1, d_model=16, H=4, L=2, num_classes=2)` — same capacity as the best SimCLR models
- The full concatenated pair sequence `(T, 1)` is fed directly into the transformer; no siamese splitting or pretrained backbone
- Classification head: linear `d_model → 2` with CrossEntropyLoss (same / different)

## Data
Same IY011/IY014 static loaders as `IY029_simclr_svm_pairwise.ipynb`. Each loader sample is two concatenated trajectories of shape `(T, 1)`, fed directly as a single sequence.

## Memory note
Full pair length T (up to ~5821 tokens for CV IY014) means O(T²) attention during backprop.
BATCH_SIZE=2 keeps peak attention memory comparable to the siamese approach at BATCH_SIZE=8
(both ≈ 1 GB per forward pass: 2 × 4 × 5821² × 4 B ≈ 1.1 GB vs. 8 × 4 × 2910² × 4 B ≈ 1.1 GB).

## Comparison
Results are plotted alongside the best SimCLR + SVM and IY025 raw-SVM baselines.

In [ ]:
import os
# Reduce CUDA allocator fragmentation — set before importing torch
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
sys.path.insert(0, '../../src')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

from dataloaders import load_loader_from_disk
from models.transformer import TransformerClassifier

plt.rcParams.update({
    'font.family': 'sans-serif', 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'axes.titlesize': 14,
})

EXP_DIR    = Path('../..')
IY011_ROOT = EXP_DIR / 'experiments/EXP-25-IY011'
IY014_ROOT = EXP_DIR / 'experiments/EXP-26-IY014'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Architecture matching the best SimCLR models (IY017)
D_MODEL    = 16
NHEAD      = 4
NUM_LAYERS = 2
DROPOUT    = 0.1
# BATCH_SIZE=2: full pair T ≈ 2×T_half → O(T²) attention ≈ same memory as siamese at B=8
BATCH_SIZE = 2
N_EPOCHS   = 100
LR         = 1e-3

print(f'Device: {DEVICE}')

DATASET_CONFIGS = [
    {
        'name':        'Baseline',
        'iy011_train': IY011_ROOT / 'data'               / 'IY011_static_train.pt',
        'iy011_val':   IY011_ROOT / 'data'               / 'IY011_static_val.pt',
        'iy011_test':  IY011_ROOT / 'data'               / 'IY011_static_test.pt',
        'iy014_train': IY014_ROOT / 'data'               / 'IY014_static_train.pt',
        'iy014_val':   IY014_ROOT / 'data'               / 'IY014_static_val.pt',
        'iy014_test':  IY014_ROOT / 'data'               / 'IY014_static_test.pt',
    },
    {
        'name':        'Mu',
        'iy011_train': IY011_ROOT / 'data_mu_variation'  / 'IY011_static_train.pt',
        'iy011_val':   IY011_ROOT / 'data_mu_variation'  / 'IY011_static_val.pt',
        'iy011_test':  IY011_ROOT / 'data_mu_variation'  / 'IY011_static_test.pt',
        'iy014_train': IY011_ROOT / 'data_mu_variation'  / 'IY014_static_train.pt',
        'iy014_val':   IY011_ROOT / 'data_mu_variation'  / 'IY014_static_val.pt',
        'iy014_test':  IY011_ROOT / 'data_mu_variation'  / 'IY014_static_test.pt',
    },
    {
        'name':        'CV',
        'iy011_train': IY011_ROOT / 'data_cv_variation'  / 'IY011_static_train.pt',
        'iy011_val':   IY011_ROOT / 'data_cv_variation'  / 'IY011_static_val.pt',
        'iy011_test':  IY011_ROOT / 'data_cv_variation'  / 'IY011_static_test.pt',
        'iy014_train': IY014_ROOT / 'data_cv_variation'  / 'IY014_static_train.pt',
        'iy014_val':   IY014_ROOT / 'data_cv_variation'  / 'IY014_static_val.pt',
        'iy014_test':  IY014_ROOT / 'data_cv_variation'  / 'IY014_static_test.pt',
    },
    {
        'name':        'T_ac',
        'iy011_train': IY011_ROOT / 'data_t_ac_variation' / 'IY011_static_train.pt',
        'iy011_val':   IY011_ROOT / 'data_t_ac_variation' / 'IY011_static_val.pt',
        'iy011_test':  IY011_ROOT / 'data_t_ac_variation' / 'IY011_static_test.pt',
        'iy014_train': IY014_ROOT / 'data_t_ac_variation' / 'IY014_static_train.pt',
        'iy014_val':   IY014_ROOT / 'data_t_ac_variation' / 'IY014_static_val.pt',
        'iy014_test':  IY014_ROOT / 'data_t_ac_variation' / 'IY014_static_test.pt',
    },
]
DS_NAMES = [cfg['name'] for cfg in DATASET_CONFIGS]

Device: cuda


/home/ianyang/micromamba/envs/stochastic_sim/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model & Helpers

In [ ]:
def load_split(pt_path):
    """Load a .pt static loader and return (X_np, y_np) as numpy arrays."""
    loader = load_loader_from_disk(pt_path, batch_size=2048)
    Xs, ys = [], []
    for X, y in loader:
        Xs.append(X.numpy())
        ys.append(y.numpy().ravel())
    # y as int64 for CrossEntropyLoss
    return np.concatenate(Xs), np.concatenate(ys).astype(np.int64)


def make_tensor_loader(X_np, y_np, batch_size, shuffle=True):
    dataset = torch.utils.data.TensorDataset(
        torch.tensor(X_np, dtype=torch.float32),
        torch.tensor(y_np, dtype=torch.long),
    )
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def evaluate(model, loader):
    """Return accuracy on a DataLoader of (X, y_long) pairs."""
    model.eval()
    correct = total = 0
    for X, y in loader:
        logits  = model(X.to(DEVICE))          # (B, 2)
        preds   = logits.argmax(dim=1)
        correct += (preds == y.to(DEVICE)).sum().item()
        total   += len(y)
    return correct / total


def train_and_eval(X_tr, y_tr, X_va, y_va, X_te, y_te, verbose=False):
    """
    Train a fresh TransformerClassifier on the full concatenated pair sequence
    and return (test_accuracy, training_loss_curve).

    The full pair (T, 1) is fed directly — no splitting into halves.
    CrossEntropyLoss with num_classes=2 (0=different, 1=same).
    """
    model = TransformerClassifier(
        input_size=1, d_model=D_MODEL, nhead=NHEAD,
        num_layers=NUM_LAYERS, num_classes=2, dropout=DROPOUT,
    ).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=N_EPOCHS)

    tr_loader = make_tensor_loader(X_tr, y_tr, BATCH_SIZE, shuffle=True)
    va_loader = make_tensor_loader(X_va, y_va, BATCH_SIZE, shuffle=False)
    te_loader = make_tensor_loader(X_te, y_te, BATCH_SIZE, shuffle=False)

    best_va, best_state = 0.0, None
    loss_curve = []

    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for X, y in tr_loader:
            optimiser.zero_grad()
            loss = criterion(model(X.to(DEVICE)), y.to(DEVICE))
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
        scheduler.step()
        loss_curve.append(epoch_loss / len(tr_loader))

        va_acc = evaluate(model, va_loader)
        if va_acc > best_va:
            best_va = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and (epoch + 1) % 20 == 0:
            print(f'  epoch {epoch+1:3d}  loss={loss_curve[-1]:.4f}  val={va_acc:.3f}')

    model.load_state_dict(best_state)
    te_acc = evaluate(model, te_loader)
    return te_acc, loss_curve

## Load Data & Train

In [ ]:
results    = {}   # {ds_name: {'iy011': acc, 'iy014': acc}}
loss_curves = {}

for cfg in DATASET_CONFIGS:
    name = cfg['name']
    results[name]     = {}
    loss_curves[name] = {}
    print(f'\n=== {name} ===')

    for fold in ('iy011', 'iy014'):
        print(f'  {fold}:', end=' ', flush=True)
        X_tr, y_tr = load_split(cfg[f'{fold}_train'])
        X_va, y_va = load_split(cfg[f'{fold}_val'])
        X_te, y_te = load_split(cfg[f'{fold}_test'])

        acc, lc = train_and_eval(X_tr, y_tr, X_va, y_va, X_te, y_te, verbose=True)
        results[name][fold]     = acc
        loss_curves[name][fold] = lc
        print(f'test acc = {acc:.4f}')

print('\nDone.')


=== Baseline ===
  iy011: 📂 Loading static data from ../../experiments/EXP-25-IY011/data/IY011_static_train.pt...
📂 Loading static data from ../../experiments/EXP-25-IY011/data/IY011_static_val.pt...
📂 Loading static data from ../../experiments/EXP-25-IY011/data/IY011_static_test.pt...
  epoch  20  loss=0.4193  val=0.765
  epoch  40  loss=0.3523  val=0.742


In [ ]:
import json

_save_path = Path('IY029_transformer_pairwise_results.json')
with open(_save_path, 'w') as _f:
    json.dump(
        {ds: {fold: float(results[ds][fold]) for fold in ('iy011', 'iy014')}
         for ds in DS_NAMES},
        _f, indent=2,
    )
print(f'Saved {_save_path}')

## Results

In [ ]:
palette   = sns.color_palette('colorblind')
ds_colors = {n: palette[i] for i, n in enumerate(DS_NAMES)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, fold, fold_title in [
    (axes[0], 'iy011', 'IY011 (2-fold)'),
    (axes[1], 'iy014', 'IY014 (10-fold)'),
]:
    for name in DS_NAMES:
        ax.plot(loss_curves[name][fold], color=ds_colors[name], label=name, lw=1.5)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('CrossEntropy loss', fontsize=12)
    ax.set_title(fold_title, fontsize=13)
    ax.legend(fontsize=10)
fig.suptitle('Training loss curves — TransformerClassifier (full pair input)', fontsize=14)
plt.tight_layout()
plt.savefig('IY029_transformer_pairwise_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
colors = [palette[i] for i in range(len(DS_NAMES))]

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
x     = np.arange(len(DS_NAMES))
width = 0.5

for ax, fold_key, fold_title in [
    (axes[0], 'iy011', 'IY011 (2-fold)'),
    (axes[1], 'iy014', 'IY014 (10-fold)'),
]:
    accs = [results[n][fold_key] for n in DS_NAMES]
    bars = ax.bar(x, accs, width, color=colors, edgecolor='black', linewidth=0.6)
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=10)
    ax.axhline(0.5, color='dimgrey', linestyle='--', linewidth=1.2, label='Chance (50%)')
    ax.set_xticks(x)
    ax.set_xticklabels(DS_NAMES, fontsize=11)
    ax.set_xlabel('Varied statistic', fontsize=12)
    ax.set_ylim(0, 1.25)
    ax.set_title(fold_title, fontsize=13)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='y', linestyle=':', alpha=0.4)

axes[0].set_ylabel('Test accuracy', fontsize=12)
fig.suptitle('TransformerClassifier — pairwise same/different accuracy\n'
             'full pair sequence, trained end-to-end',
             fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig('IY029_transformer_pairwise_acc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved IY029_transformer_pairwise_acc.png')

In [ ]:
import pandas as pd
rows = []
for n in DS_NAMES:
    rows.append({
        'Dataset':         n,
        '2-fold (IY011)':  results[n]['iy011'],
        '10-fold (IY014)': results[n]['iy014'],
        'Mean':            np.mean([results[n]['iy011'], results[n]['iy014']]),
    })
df = pd.DataFrame(rows).set_index('Dataset')
pd.set_option('display.float_format', '{:.3f}'.format)
print(df.to_string())